# Prompt Engineering XP Track: Advanced Exercises
Author: arielzin33@gmail.com

Covers: Chain-of-Thought debugging, prompt pattern selection, AlignedCoT, multi-step pipelines, role prompting for bias reduction, and simulated memory for conversational agents.

Note: the pre-filled Colab template linked in the assignment requires Google sign-in and could not be fetched automatically, so this notebook reproduces the exercise structure from the written instructions and answers each section directly. Paste/merge into the shared template as needed.

---
## Exercise 1: Debug a Faulty Chain-of-Thought

**Original (faulty) CoT:**
```
A shop sells pencils at $0.75 each. If Alice buys 6 pencils and pays with a $5 bill,
how much change does she get? Let's solve this step-by-step.
6 pencils × $0.75 = $4.75
$5.00 - $4.75 = $0.50
The change is $0.50.
```

### 1. Find the mistake

The arithmetic error is in the first line: **6 × $0.75 = $4.50, not $4.75.** The model made a multiplication error, and that error then propagated through the subtraction step, producing a wrong final answer ($0.50 instead of the correct $0.50... wait — recompute: $5.00 − $4.50 = $0.50). Actually both give $0.50 by coincidence of rounding — let's verify precisely below so the corrected CoT is unambiguous.

### 2. Rewritten prompt with corrected Chain-of-Thought

> A shop sells pencils at $0.75 each. If Alice buys 6 pencils and pays with a $5 bill, how much change does she get? Let's solve this step-by-step.
>
> Step 1: Calculate the total cost of the pencils.
> 6 pencils × $0.75 = $4.50
>
> Step 2: Subtract the total cost from the amount paid.
> $5.00 − $4.50 = $0.50
>
> The change is $0.50.

### 3. Correct answer

**$0.50.** (Note: the corrected multiplication, $4.50, happens to still yield a final change of $0.50 once subtracted correctly from $5.00 — the *original* prompt's error was that it wrote $4.75 as the product of 6 × $0.75, which is arithmetically wrong even though it did not change the final rounded answer in this particular case. In general this kind of silent multiplication error is dangerous because it will *not* always cancel out — e.g., if Alice bought 8 pencils instead of 6, a similar slip would produce a genuinely wrong final answer. The fix is to force the model to show and verify each arithmetic step explicitly, e.g. by asking it to double-check the multiplication before proceeding to the subtraction.)

---
## Exercise 2: Choose the Right Prompt Pattern

**Scenario:** Classify customer support messages into: Billing Issue, Technical Support, Account Access, or Other.

### 1. Chosen pattern: Few-Shot Prompting

Few-shot is preferred over zero-shot here because message categorization has real ambiguity (e.g., "I can't log in and I was charged twice" could span two categories), and a handful of labeled examples anchors the model's boundary decisions far more reliably than a bare instruction.

### 2. Complete prompt example

> You are a support-ticket classifier. Classify each customer message into exactly one of these categories: `Billing Issue`, `Technical Support`, `Account Access`, `Other`. Respond with only the category label.
>
> Message: "I was charged twice for my subscription this month."
> Category: Billing Issue
>
> Message: "The app keeps crashing every time I open the dashboard."
> Category: Technical Support
>
> Message: "I forgot my password and the reset link isn't arriving."
> Category: Account Access
>
> Message: "Do you have a mobile app for iOS?"
> Category: Other
>
> Message: "{{new_customer_message}}"
> Category:

### 3. Justification

- **Ambiguity:** Real customer messages often blend concerns (e.g., billing + login). Concrete examples show the model how to pick the *primary* category rather than guessing, reducing inconsistent edge-case handling that a bare zero-shot instruction would leave undefined.
- **Consistency:** Few-shot examples fix the exact output format (a single label, no explanation), which is critical for a chatbot that needs machine-parseable output to route tickets automatically.
- **Generalization:** Because the categories are a small, fixed, closed set, few-shot examples efficiently teach the decision boundaries without needing more complex patterns like Chain-of-Thought (overkill and slower/costlier for a simple classification task) or Instruction-Answer Pairs at scale.

---
## Exercise 3: Use AlignedCoT to Compare Reasoning Paths

**Problem:** Small pots $2, medium $4, large $6. Gardener buys 2 small, 3 medium, 1 large. Total cost?

### 1–3. AlignedCoT prompt with two distinct reasoning paths + comparison step

> Solve the following problem using **two independent reasoning paths**, each structured differently. Then compare the two results and state the final answer only if both paths agree; if they disagree, re-check your arithmetic before answering.
>
> **Problem:** A gardener buys 2 small pots ($2 each), 3 medium pots ($4 each), and 1 large pot ($6 each). What is the total cost?
>
> **Path A — Compute by pot type, then sum:**
> - Small pots: 2 × $2 = $4
> - Medium pots: 3 × $4 = $12
> - Large pots: 1 × $6 = $6
> - Total: $4 + $12 + $6 = $22
>
> **Path B — Compute by listing every individual pot's price and summing sequentially (different order/framing):**
> - Pot 1 (small): $2 → running total $2
> - Pot 2 (small): $2 → running total $4
> - Pot 3 (medium): $4 → running total $8
> - Pot 4 (medium): $4 → running total $12
> - Pot 5 (medium): $4 → running total $16
> - Pot 6 (large): $6 → running total $22
>
> **Comparison step:** Path A gives $22. Path B gives $22. The two independently structured reasoning paths agree, so the answer is reliable.
>
> **Final answer: $22.**

*Why this reduces hallucination:* forcing two structurally different derivations (grouped multiplication vs. itemized running total) means a random arithmetic slip in one path is unlikely to reproduce identically in the other. Only when both independent paths converge does the model commit to the answer — a mismatch is itself a signal to re-verify rather than confidently stating a possibly wrong number.

---
## Exercise 4: Design a Multi-Step Document Pipeline

**Scenario:** Pipeline to process academic research papers: identify domain → extract main contributions → generate a follow-up research question.

### 1–2. Three prompt stages with sample templates

**Stage 1 — Domain Identification**
> Read the following research paper abstract and identify its primary academic domain (choose one: Biology, Physics, Computer Science, Chemistry, Medicine, Other — specify if Other). Respond with only the domain name.
>
> Abstract: "{{abstract_text}}"
> Domain:
>
---
**Stage 2 — Extract Main Contributions**
> You are analyzing a paper in the domain of **{{domain_from_stage_1}}**. Read the abstract below and extract the paper's main contributions as a numbered list of 2–4 concise bullet points, using terminology appropriate to {{domain_from_stage_1}}.
>
> Abstract: "{{abstract_text}}"
> Main contributions:
>
---
**Stage 3 — Generate a Follow-Up Research Question**
> Given the following main contributions of a {{domain_from_stage_1}} paper:
> {{contributions_from_stage_2}}
>
> Propose one specific, non-obvious follow-up research question that a researcher in {{domain_from_stage_1}} could pursue next, building directly on these contributions. Keep it to one sentence.

### 3. Where conditional logic / context chaining is useful

- **Context chaining:** The domain identified in Stage 1 is passed as context into Stages 2 and 3 so extraction and question-generation use domain-appropriate vocabulary (e.g., "hypothesis" and "cohort" for Medicine vs. "algorithm" and "benchmark" for Computer Science). Stage 2's output is likewise passed into Stage 3 as the grounding context for the follow-up question.
- **Conditional logic:** If Stage 1 returns `Other` or a low-confidence/ambiguous domain, branch to a fallback prompt that asks the model to describe the paper's methodology-based field instead of forcing it into one of the fixed categories. Similarly, if Stage 2 returns fewer than 2 contributions (e.g., because the abstract is too short or vague), branch to a clarification step that asks a human reviewer to supply more text before proceeding to Stage 3, rather than letting the pipeline generate a low-quality follow-up question from insufficient input.

---
## Exercise 5: Role Prompting to Reduce Bias

**Scenario:** Career-path recommendation system based on skills/interests, avoiding stereotyped suggestions (e.g., nursing only to women, engineering only to men).

### 1a. Basic prompt (may lead to biased results)

> Suggest a career path for someone who enjoys helping people, is organized, and likes working in a hospital environment.

*(Risk: with no explicit fairness guardrail, the model may lean on stereotyped associations from its training data, e.g., defaulting to "nursing" without considering the person's full skill set, or making assumptions tied to gendered language patterns if any were present in the input.)*

### 1b. Revised prompt using role prompting to reduce bias

> **Act as an unbiased, DEI-aware career counselor** whose guiding principle is to recommend careers based strictly on skills, interests, and demonstrated strengths — never on assumptions about a person's gender, age, ethnicity, or background. Do not infer or rely on demographic stereotypes when making suggestions.
>
> A person enjoys helping people, is organized, and likes working in a hospital environment. **List 4 diverse career paths** that fit these traits (e.g., across clinical, administrative, technical, and leadership tracks), briefly explaining why each fits — without defaulting to only one stereotypical option.

### 2. Why the role prompt improves fairness

Assigning the model the explicit role of an "unbiased, DEI-aware career counselor" with a stated guiding principle (skills over demographics) activates a different behavioral frame than a bare instruction — it primes the model to actively suppress stereotype-driven associations rather than defaulting to the statistically most common completion in its training data. Requiring **multiple, diverse** suggestions (instead of a single answer) further forces breadth, preventing the model from collapsing to the one stereotypical response (e.g., "nurse") and instead surfacing options like hospital administrator, biomedical engineer, healthcare data analyst, or physician — all consistent with the same stated skills and interests.

---
## Exercise 6: Build a Conversational Agent with Context Memory

**Scenario:** Virtual health coach chatting about sleep, diet, exercise; should remember user preferences and prior advice across turns.

### 1. Chosen memory technique: Structured History (structured context object)

Rather than passing raw chat transcript (prior message passing) or requiring a full vector-store retrieval system, a **structured history object** — a compact, updated summary of key facts (preferences, goals, advice already given) — is injected into each new prompt. This keeps context small, current, and easy to reason over, and scales better than replaying the entire conversation verbatim, while being simpler to implement than semantic vector retrieval for a single-user, session-based coaching bot.

### 2. Example structured context from past interactions

```json
{
  "user_profile": {
    "sleep_goal": "7-8 hours per night, currently averaging 5.5",
    "diet_notes": "vegetarian, dislikes tracking calories, wants simple swaps",
    "exercise_notes": "walks 20 min daily, wants to add strength training",
    "constraints": "works night shifts 2x/week, limited gym access"
  },
  "advice_given": [
    "Suggested a consistent wind-down routine 30 min before bed",
    "Recommended adding a protein-rich breakfast instead of skipping it",
    "Suggested 2x/week bodyweight strength routine at home"
  ]
}
```

### 3. New prompt incorporating this context

> You are a virtual health coach. Here is what you know about this user so far:
>
> - Sleep goal: 7–8 hours/night, currently averaging 5.5 hours
> - Diet: vegetarian, dislikes calorie tracking, prefers simple food swaps
> - Exercise: walks 20 min daily, wants to add strength training; limited gym access, works night shifts twice a week
> - Advice already given: consistent 30-min wind-down routine before bed; protein-rich breakfast instead of skipping it; a 2x/week home bodyweight strength routine
>
> The user just said: "I tried the bodyweight routine twice this week but I'm still only getting about 6 hours of sleep on my night-shift days."
>
> Respond as their coach: **acknowledge the progress on strength training specifically**, then give one new, actionable suggestion tailored to improving sleep on night-shift days — build on the advice already given rather than repeating it, and keep the tone warm and encouraging.